In [1]:
import numpy as np 



In [2]:
data_1e6 = np.load('/export/data/vgiusepp/odisseo_data/data_fix_position/sbi-sim/data/sbi-benchmarks/odisseo/x_1000000.npy')

In [3]:
!pip install concurrent.futures

ERROR: Could not find a version that satisfies the requirement concurrent.futures (from versions: none)
ERROR: No matching distribution found for concurrent.futures


In [4]:
from concurrent.futures import ThreadPoolExecutor, as_completed


In [5]:
def process_sample(sample):
    """Compute 3 histograms from 1 sample"""
    bins = [64, 32]

    ph1_phi2, _, _ = np.histogram2d(sample[1], sample[2], bins=bins, range=[[-120., 70.], [-8, 2]])
    R_vR, _, _ = np.histogram2d(sample[0], sample[3], bins=bins, range=[[6., 20.], [-250., 250.]])
    vphicosphi2_vphi2, _, _ = np.histogram2d(sample[4], sample[5], bins=bins, range=[[-2., 1.], [-0.1, 0.1]])

    return np.stack([ph1_phi2, R_vR, vphicosphi2_vphi2], axis=0)  # Shape: (3, 64, 32)


def calculate_histogram_stats_parallel(data, max_workers=8):
    """Parallel histogram stats computation: mean/std over log1p histograms"""
    all_histograms = []

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = [executor.submit(process_sample, sample) for sample in data]
        for future in as_completed(futures):
            all_histograms.append(future.result())

    all_histograms = np.stack(all_histograms)  # Shape: (N_samples, 3, 64, 32)
    all_histograms = np.log1p(all_histograms)

    # Calculate mean and std for each channel independently
    mean_per_channel = np.mean(all_histograms, axis=(0, 2, 3))  # Shape: (3,)
    std_per_channel = np.std(all_histograms, axis=(0, 2, 3))    # Shape: (3,)

    return mean_per_channel, std_per_channel

# Usage in your notebook:
mean_histogram_1e6, std_histogram_1e6 = calculate_histogram_stats_parallel(data_1e6)
np.savez('/export/data/vgiusepp/odisseo_data/data_fix_position/preprocess/mean_std_log_1e6.npz', 
         mean_x=mean_histogram_1e6, 
         std_x=std_histogram_1e6)

In [6]:
print('mean histogram', mean_histogram_1e6)
print('std histogram', std_histogram_1e6)

mean histogram [0.00098003 0.00035404 0.00034501]
std histogram [0.02686971 0.0156627  0.01547433]


In [11]:
# Calculate statistics for each histogram type separately
def calculate_histogram_stats(data):
    """Calculate mean and std for each histogram channel independently"""
    bins = [64, 32]
    all_histograms = []
    
    for sample in data:
        ph1_phi2, _, _ = np.histogram2d(sample[1], sample[2], bins=bins, range=[[-120., 70.], [-8, 2]])
        R_vR, _, _ = np.histogram2d(sample[0], sample[3], bins=bins, range=[[6., 20.], [-250., 250.]])
        vphicosphi2_vphi2, _, _ = np.histogram2d(sample[4], sample[5], bins=bins, range=[[-2., 1.], [-0.1, 0.1]])
        histograms = np.stack([ph1_phi2, R_vR, vphicosphi2_vphi2], axis=0)
        all_histograms.append(histograms)
    
    # all_histograms = np.log1p(all_histograms)  # Shape: (N_samples, 3, 64, 32)
    
    # Calculate mean and std for each channel independently
    mean_per_channel = np.mean(all_histograms, axis=(0, 2, 3))  # Shape: (3,)
    std_per_channel = np.std(all_histograms, axis=(0, 2, 3))    # Shape: (3,)

    return mean_per_channel, std_per_channel

# Usage in your notebook:
# mean_histogram_1e6_2, std_histogram_1e6_2 = calculate_histogram_stats(data_1e6)
np.savez('/export/data/vgiusepp/odisseo_data/data_fix_position/preprocess/mean_std_channel_1e6.npz', 
         mean_x=mean_histogram_1e6_2, 
         std_x=std_histogram_1e6_2)

In [10]:
print('mean histogram', mean_histogram_1e6_2)
print('std histogram', std_histogram_1e6_2)

mean histogram [0.00145455 0.00051081 0.00049815]
std histogram [0.04060171 0.02259976 0.02235672]


In [5]:
print('hello')

hello


In [4]:

print(mean_histogram_1e6.shape, std_histogram_1e6.shape)

(3, 64, 32) (3, 64, 32)


In [5]:
data_1e5 = np.load('/export/data/vgiusepp/odisseo_data/data_fix_position/sbi-sim/data/sbi-benchmarks/odisseo/x_100000.npy')

mean_histogram_1e5, std_histogram_1e5 = calculate_histogram_stats(data_1e5)

print(mean_histogram_1e5.shape, std_histogram_1e5.shape)

(3, 64, 32) (3, 64, 32)


In [18]:
(mean_histogram_1e6 - mean_histogram_1e5).mean()

np.float64(6.445312499999983e-07)

In [17]:
(std_histogram_1e6 - std_histogram_1e5).mean()

np.float64(0.00042155051943389236)

In [1]:
import matplotlib.pyplot as plt
plt.hist(std_histogram_1e6.flatten(), bins=100, range=(0, 0.5))

NameError: name 'std_histogram_1e6' is not defined

In [4]:
import jax.numpy as jnp

mean_x = jnp.load('/export/data/vgiusepp/odisseo_data/data_fix_position/preprocess/mean_std_1e6.npz')['std_x']
print(mean_x)

[[[0.00387295 0.00374163 0.00299999 ... 0.00299999 0.00374163 0.00282842]
  [0.00360553 0.00244948 0.00282842 ... 0.00244948 0.00346408 0.00331661]
  [0.00435886 0.00299999 0.00360553 ... 0.00374163 0.00346408 0.00282842]
  ...
  [0.00264574 0.00223606 0.002      ... 0.00282842 0.00264574 0.00244948]
  [0.00282842 0.00264574 0.00223606 ... 0.002      0.00316226 0.00223606]
  [0.00223606 0.001      0.00173205 ... 0.00173205 0.00223606 0.002     ]]

 [[0.         0.         0.00223606 ... 0.00141421 0.00141421 0.001     ]
  [0.         0.         0.00141421 ... 0.001      0.001      0.        ]
  [0.         0.         0.00244948 ... 0.00141421 0.         0.        ]
  ...
  [0.         0.001      0.002      ... 0.001      0.         0.        ]
  [0.         0.001      0.00173205 ... 0.00141421 0.         0.        ]
  [0.         0.001      0.002      ... 0.         0.         0.        ]]

 [[0.00316226 0.00360553 0.00299999 ... 0.00173205 0.00331661 0.00244948]
  [0.00264574 0.003464